# Продовження навчання з чекпоінта

Завантажує модель і `VecNormalize` з чекпоінта і доводить навчання до `TOTAL_STEPS`.
`config.py` і `hexapod_env.py` не мають змінюватися з моменту старту цього запуску, інакше друга частина піде на іншій нагороді.

In [1]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

from hexapod_env import HexapodEnv

In [2]:
N_ENVS = 4
TOTAL_STEPS = 5_000_000
CKPT = 3_000_000
RUN_NAME = "stage3.3-sway"
CKPT_DIR = f"models/{RUN_NAME}/checkpoints"

In [3]:
def make_env():
    return Monitor(HexapodEnv())

vec_env = SubprocVecEnv([make_env for _ in range(N_ENVS)])
vec_env = VecNormalize.load(f"{CKPT_DIR}/rl_model_vecnormalize_{CKPT}_steps.pkl", vec_env)
vec_env.training = True       # load за замовчуванням вимикає оновлення статистики
vec_env.norm_reward = True

In [4]:
INFO_KEYS = (
    "forward_vel",
    "side_vel",
    "heading_err_deg",
    "reward",
    "r_vel",
    "r_gait",
    "r_heading",
    "r_vy",
    "r_air",
    "pen_roll",
    "pen_pitch",
    "pen_stroke",
    "pen_angvel",
    "pen_vy",
    "pen_vz",
    "pen_rate",
)


class InfoCallback(BaseCallback):
    def _on_step(self):
        for info in self.locals["infos"]:
            for k in INFO_KEYS:
                self.logger.record_mean("env/" + k, info[k])
        return True

In [5]:
lr = lambda progress: 3e-5

model = PPO.load(
    f"{CKPT_DIR}/rl_model_{CKPT}_steps",
    env=vec_env,
    device="cpu",
    tensorboard_log="logs/",
    custom_objects={"learning_rate": lr, "lr_schedule": lr},
)
print("кроків у чекпоінті:", model.num_timesteps)

checkpoint = CheckpointCallback(
    save_freq=100_000 // N_ENVS,
    save_path=CKPT_DIR,
    save_vecnormalize=True,
)

кроків у чекпоінті: 3000000


In [6]:
model.learn(
    total_timesteps=TOTAL_STEPS - CKPT,
    callback=[checkpoint, InfoCallback()],
    tb_log_name=RUN_NAME,
    reset_num_timesteps=False,
)

model.save(f"models/{RUN_NAME}/model")
vec_env.save(f"models/{RUN_NAME}/vecnormalize.pkl")
vec_env.close()

Logging to logs/stage3.3-sway_1
height=0.1123, gravity_z=-1.0000, terminated=False, truncated=True
height=0.1127, gravity_z=-1.0000, terminated=False, truncated=True
height=0.1129, gravity_z=-1.0000, terminated=False, truncated=True
height=0.1123, gravity_z=-1.0000, terminated=False, truncated=True
--------------------------------------
| env/               |               |
|    forward_vel     | 0.169         |
|    heading_err_deg | 0.4685507     |
|    pen_angvel      | 0.053688806   |
|    pen_pitch       | 4.16407e-05   |
|    pen_rate        | 3.2           |
|    pen_roll        | 4.1174775e-05 |
|    pen_stroke      | 0.00102       |
|    pen_vy          | 0.000635      |
|    pen_vz          | 0.000721      |
|    r_air           | 0.0156        |
|    r_gait          | 0.878         |
|    r_heading       | 0.9794704     |
|    r_vel           | 0.975         |
|    r_vy            | 0.871         |
|    reward          | 4.35          |
|    side_vel        | 0.00057       